## Spaceship Titanic - Nicolas Zafred Paiva

### 1. Lets bring the librarys to this notebook:

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import OneHotEncoder #Function for categorical rows
from sklearn.model_selection import train_test_split #Function to split the data into 
from sklearn.neural_network import MLPClassifier #Function that implements a multi-layer perceptron (MLP)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier

from pre_process_data import pre_data # Library to processing the dataset

from sklearn.metrics import confusion_matrix, classification_report

from alarm_buzz import buzz, message_telegram_bot # Comand for the noise and message


### 2. First, loading the data:

In [2]:
df = pd.read_csv('train.csv') #Training set

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8693 entries, 0 to 8692
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   8693 non-null   str    
 1   HomePlanet    8492 non-null   str    
 2   CryoSleep     8476 non-null   object 
 3   Cabin         8494 non-null   str    
 4   Destination   8511 non-null   str    
 5   Age           8514 non-null   float64
 6   VIP           8490 non-null   object 
 7   RoomService   8512 non-null   float64
 8   FoodCourt     8510 non-null   float64
 9   ShoppingMall  8485 non-null   float64
 10  Spa           8510 non-null   float64
 11  VRDeck        8505 non-null   float64
 12  Name          8493 non-null   str    
 13  Transported   8693 non-null   bool   
dtypes: bool(1), float64(6), object(2), str(5)
memory usage: 1.2+ MB


In [4]:
# Lets test the functions to prepare the dataset:
df_final = pre_data(df)
df.head(5)

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported,avg_spend
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False,0.0
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True,147.2
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False,2076.6
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False,1035.2
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True,218.2


In [5]:
df_final.head(5)

,CryoSleep,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Transported,avg_spend,...,Deck_A,Deck_B,Deck_C,Deck_D,Deck_E,Deck_F,Deck_G,Deck_T,Side_P,Side_S
0,0,39.0,0,0.0,0.0,0.0,0.0,0.0,0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,0,24.0,0,109.0,9.0,25.0,549.0,44.0,1,147.2,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
2,0,58.0,1,43.0,3576.0,0.0,6715.0,49.0,0,2076.6,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,0,33.0,0,0.0,1283.0,371.0,3329.0,193.0,0,1035.2,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,0,16.0,0,303.0,70.0,151.0,565.0,2.0,1,218.2,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0


### 3. Spliting the data into train, cv and test:

In [6]:
X = df_final.drop(columns = 'Transported')
y = df_final['Transported']

In [ ]:

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.10, random_state=10, stratify=y)

print("X_train.shape", X_train.shape, "y_train.shape", y_train.shape)
print("X_test.shape", X_test.shape, "y_test.shape", y_test.shape)

X_train.shape (8258, 26) y_train.shape (8258,)
X_test.shape (435, 26) y_test.shape (435,)


### 4. Extreme Gradient Boosted Trees Model (XGBoost)

In [ ]:
# In this cell we are goint to perform a grid search using de XGBoost tree algorithm:
rf_xg = XGBClassifier(random_state = 47)

param_grid = {
    'n_estimators': [100, 200, 300, 500],
    'learning_rate': [0.01, 0.05, 0.1, 1.0],
    'max_depth': [3, 4, 5, 6, 8, 10],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.7, 0.8, 1.0]
}

grid_search = GridSearchCV(
    estimator = rf_xg,
    param_grid = param_grid,
    cv = 5, #number of folders for cross validation
    scoring = 'accuracy',
    n_jobs = -1 #controls how many CPU cores can be used in parallel, =-1 mean all!
)

grid_search.fit(X_train, y_train)

buzz(sett = 1) # Make the noise when the grid search is completed
# Message to my cell bot
message_telegram_bot(f"Grid Search finished!\nBest parameters: {grid_search.best_params_}\nbest score:{grid_search.best_score_}")

In [21]:
y_pred = grid_search.predict(X_test)

print(np.mean(y_pred == y_test))

0.7839080459770115


Lets see if we can make this model even better:

In [22]:
# Here we are going to see the confusion matrix

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

[[166  50]
 [ 44 175]]
              precision    recall  f1-score   support

           0       0.79      0.77      0.78       216
           1       0.78      0.80      0.79       219

    accuracy                           0.78       435
   macro avg       0.78      0.78      0.78       435
weighted avg       0.78      0.78      0.78       435



In [23]:
# lets look to the CV results:
print(grid_search.best_params_)
print(grid_search.best_score_)

{'colsample_bytree': 1.0, 'learning_rate': 0.1, 'max_depth': 4, 'n_estimators': 300, 'subsample': 1.0}
0.8169034688786457


In [24]:
results = pd.DataFrame(grid_search.cv_results_)

results[
    ['params', 'mean_test_score', 'std_test_score', 'rank_test_score']
].sort_values('rank_test_score').head(10)

,params,mean_test_score,std_test_score,rank_test_score
493,"{'colsample_bytree': 1.0, 'learning_rate': 0.1...",0.816903,0.010535,1
301,"{'colsample_bytree': 0.8, 'learning_rate': 0.1...",0.815693,0.010239,2
63,"{'colsample_bytree': 0.7, 'learning_rate': 0.0...",0.815692,0.007536,3
108,"{'colsample_bytree': 0.7, 'learning_rate': 0.1...",0.815208,0.008134,4
111,"{'colsample_bytree': 0.7, 'learning_rate': 0.1...",0.815087,0.008388,5
109,"{'colsample_bytree': 0.7, 'learning_rate': 0.1...",0.814966,0.007957,6
447,"{'colsample_bytree': 1.0, 'learning_rate': 0.0...",0.814844,0.010248,7
446,"{'colsample_bytree': 1.0, 'learning_rate': 0.0...",0.814725,0.004900,8
255,"{'colsample_bytree': 0.8, 'learning_rate': 0.0...",0.814725,0.007388,9
307,"{'colsample_bytree': 0.8, 'learning_rate': 0.1...",0.814724,0.007747,10


In [17]:
model = grid_search.best_estimator_

importance = model.feature_importances_

for feature, value in zip(X_train.columns, importance):
    print(feature, value)

CryoSleep 0.21078542
Age 0.0141401375
VIP 0.012239371
RoomService 0.035353784
FoodCourt 0.02782991
ShoppingMall 0.023414511
Spa 0.036179677
VRDeck 0.030451568
avg_spend 0.057990763
group_pas 0.01468552
num_group_pas 0.0122427
Cabin_num 0.018259443
HomePlanet_Earth 0.10995412
HomePlanet_Europa 0.04388457
HomePlanet_Mars 0.01935018
Destination_55 Cancri e 0.015871447
Destination_PSO J318.5-22 0.013324464
Destination_TRAPPIST-1e 0.01841751
Deck_A 0.016207702
Deck_B 0.017972056
Deck_C 0.026582414
Deck_D 0.017249335
Deck_E 0.050028548
Deck_F 0.023422562
Deck_G 0.073622294
Deck_T 0.0
Side_P 0.023544854
Side_S 0.03699515


### 5. Submission of the predictions

In [25]:
df_test = pd.read_csv('test.csv') #Test dataset
df_submission = pd.read_csv('sample_submission.csv') #File to fill with the predictions

df_test.head(5)

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name
0,0013_01,Earth,True,G/3/S,TRAPPIST-1e,27.0,False,0.0,0.0,0.0,0.0,0.0,Nelly Carsoning
1,0018_01,Earth,False,F/4/S,TRAPPIST-1e,19.0,False,0.0,9.0,0.0,2823.0,0.0,Lerome Peckers
2,0019_01,Europa,True,C/0/S,55 Cancri e,31.0,False,0.0,0.0,0.0,0.0,0.0,Sabih Unhearfus
3,0021_01,Europa,False,C/1/S,TRAPPIST-1e,38.0,False,0.0,6652.0,0.0,181.0,585.0,Meratz Caltilter
4,0023_01,Earth,False,F/5/S,TRAPPIST-1e,20.0,False,10.0,0.0,635.0,0.0,0.0,Brence Harperez


In [26]:
df_sub = pre_data(df_test, f1=1)
df_sub.head(5)

,PassengerId,CryoSleep,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,avg_spend,...,Deck_A,Deck_B,Deck_C,Deck_D,Deck_E,Deck_F,Deck_G,Deck_T,Side_P,Side_S
0,0013_01,1,27.0,0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
1,0018_01,0,19.0,0,0.0,9.0,0.0,2823.0,0.0,566.4,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
2,0019_01,1,31.0,0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,0021_01,0,38.0,0,0.0,6652.0,0.0,181.0,585.0,1483.6,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,0023_01,0,20.0,0,10.0,0.0,635.0,0.0,0.0,129.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0


In [27]:
X_sub = df_sub.drop(columns = ['Transported', 'PassengerId'])

In [28]:
df_sub['Transported'] = grid_search.predict(X_sub)

In [29]:
df_sub['Transported'] = df_sub['Transported'].astype(bool)
df_sub.head(5)

,PassengerId,CryoSleep,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,avg_spend,...,Deck_A,Deck_B,Deck_C,Deck_D,Deck_E,Deck_F,Deck_G,Deck_T,Side_P,Side_S
0,0013_01,1,27.0,0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
1,0018_01,0,19.0,0,0.0,9.0,0.0,2823.0,0.0,566.4,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
2,0019_01,1,31.0,0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,0021_01,0,38.0,0,0.0,6652.0,0.0,181.0,585.0,1483.6,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,0023_01,0,20.0,0,10.0,0.0,635.0,0.0,0.0,129.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0


In [30]:
# Finaly:
df_sub_final = df_sub[['PassengerId', 'Transported']]

df_sub_final.to_csv("sample_submission_05.csv", index=False)

### Finis